Unique Pallets - new pallets in the flash report
=
This notebook compares two lists of module pallets and prints the pallets of the new list that aren't in the old one. It's used to find which pallets were added between two flash reports.

**Workflow overview:**
1. Read the `OLD` and `NEW_UNIQUE` sheets of the flash report workbook.
2. Normalise the `pallet` column of both (text, trimmed, upper case).
3. Keep the `NEW_UNIQUE` rows whose pallet doesn't appear in `OLD`, and print them.
4. If nothing is found, print some debugging info.

**Input prerequisites:**
- **File:** `C:\Users\Usuario\Desktop\flash reports.xlsx`.
- **Sheets:** `OLD` and `NEW_UNIQUE` (exact names).
- **Column:** both sheets have a `pallet` column (exact, case-sensitive name). The other columns of `NEW_UNIQUE` (e.g. `módulo`, `intensidade`, `mod-int`) are only printed.
- Empty rows or total rows in `NEW_UNIQUE` also show up as "new" pallets (an empty cell becomes the text `NAN`), so the sheet should hold pallet rows only.

Finding the pallets that are only in NEW
=
- Reads both sheets inside a `with pd.ExcelFile(...)` block, so the workbook is opened only once.
- `old_clean` / `new_clean`: `pallet` as text, trimmed and upper case, so `abc123 ` and `ABC123` match.
- `new_only` = rows of `NEW_UNIQUE` whose cleaned pallet isn't in `old_clean` (`~isin`). The original values are printed, not the cleaned ones.
- The result is only printed, nothing is saved.
- The debugging block only runs when nothing is found. Its NaN count is always 0: `astype(str)` already turned the empty cells into the text `nan` (`NAN` after `upper()`), so it can't find them.

In [25]:
import pandas as pd

with pd.ExcelFile(r'C:\Users\Usuario\Desktop\flash reports.xlsx') as xlsx:
    old = pd.read_excel(xlsx, sheet_name='OLD')
    new = pd.read_excel(xlsx, sheet_name='NEW_UNIQUE')

# Clean and standardize both columns
old_clean = old['pallet'].astype(str).str.strip().str.upper()
new_clean = new['pallet'].astype(str).str.strip().str.upper()

# Find values in new that aren't in old
new_only = new[~new_clean.isin(old_clean)]

print(f"Found {len(new_only)} rows in NEW that are not in OLD")
print(new_only)

# If still empty, let's investigate further
if len(new_only) == 0:
    print("\n=== DEBUGGING INFO ===")
    print(f"Old shape: {old.shape}, New shape: {new.shape}")
    print(f"Old sample: {old_clean.head().tolist()}")
    print(f"New sample: {new_clean.head().tolist()}")
    print(f"Are there NaN values? Old: {old_clean.isna().sum()}, New: {new_clean.isna().sum()}")

Found 433 rows in NEW that are not in OLD
                          pallet  módulo  intensidade mod-int
0    T01250905000356A018916E19F2   705.0          2.0   705-2
1    T01250904004266A018916E19F2   705.0          2.0   705-2
2    T01250905000066A018916E19F2   705.0          2.0   705-2
3    T01250905004206A018916E19F2   705.0          2.0   705-2
4    T01250905003946A018916E19F2   705.0          2.0   705-2
..                           ...     ...          ...     ...
428  T01250905002626A018916E19F2   715.0          2.0   715-2
429  T01250905002476A018916E19F2   715.0          2.0   715-2
430  T01250905001786A018916E19F2   715.0          2.0   715-2
431  T01250906002666A018916E19F3   715.0          3.0   715-3
432                            0     NaN          NaN     NaN

[433 rows x 4 columns]
